# 04 — Reasoning: real transit travel time, not straight-line distance

Answers the question that kicked off this chapter: the earlier "nearest
playground" demo (`03_kg_visualization.ipynb`) picked the closest POI by
straight-line distance alone. This notebook replaces that with an actual
public-transport travel-time estimate using Wiener Linien's GTFS schedule —
**now supporting up to 2 transfers**, not just direct connections (see
`docs/reasoning_layer_decisions.md` for the full history: this started as
direct-only, then got extended once direct-only's real coverage — under 1% of
POI pairs — turned out to be a bad limitation for a project about generating
activity suggestions).

The actual routing logic lives in `reasoning/gtfs_routing.py` — this notebook
loads it and pokes at the results. Two real bugs were found and fixed while
building the 2-transfer search (both documented in the module and in
`docs/reasoning_layer_decisions.md`):
1. Picking only the single nearest stop as a search origin can land on a
   badly-connected platform even when a slightly farther one boards well —
   fixed by considering every stop within walking distance.
2. The total-time formula was double-counting the initial walk leg (a small
   but real bug present even in the original direct-only version).

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter, haversine_m

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
print(f"KG loaded: {len(g)} triples")

router = GtfsRouter(date="20260815")  # a representative Saturday
print(f"GTFS router ready: {len(router.stop_times):,} stop_time rows "
      f"(filtered from 7.1M in the raw feed), {router.trips.trip_id.nunique():,} trips, "
      f"{len(router.stops):,} stops")

## 1. Do the two "nearest stop" lookups still agree?

Re-checking this after the 2-transfer rework, since the origin-side search now
considers many nearby platforms rather than a single nearest one — worth
confirming the RDF side (which still just picks one nearest `viennakg:Stop`,
used for RBL/live-data lookups) still lands in the same physical area.

In [ ]:
def nearest_rdf_stop(lon, lat):
    q = """
    PREFIX schema: <https://schema.org/>
    PREFIX viennakg: <http://example.org/viennakg#>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    SELECT ?name ?lon ?lat WHERE {
        ?stop a viennakg:Stop ; schema:name ?name ; geo:long ?lon ; geo:lat ?lat .
    }
    """
    candidates = [(str(r.name), float(r.lon), float(r.lat)) for r in g.query(q)]
    best = min(candidates, key=lambda c: haversine_m(lon, lat, c[1], c[2]))
    return {"stop_name": best[0], "lon": best[1], "lat": best[2],
            "distance_m": haversine_m(lon, lat, best[1], best[2])}

test_lon, test_lat = 16.368378925180206, 48.204590314427854  # Albertina

rdf_nearest = nearest_rdf_stop(test_lon, test_lat)
gtfs_nearest = router.nearest_stop(test_lon, test_lat)

print("Nearest viennakg:Stop (RDF, has RBL for live data):", rdf_nearest["stop_name"],
      f"({rdf_nearest['distance_m']:.0f} m)")
print("Nearest GTFS stop (schedule, for travel time):     ", gtfs_nearest["stop_name"],
      f"({gtfs_nearest['distance_m']:.0f} m)")
print("\nDistance between the two 'nearest stop' answers:",
      f"{haversine_m(rdf_nearest['lon'], rdf_nearest['lat'], gtfs_nearest['lon'], gtfs_nearest['lat']):.0f} m")

## 2. Before/after: a pair that was "impossible" under direct-only routing

Albertina → Rathauspark (parks) was one of the deliberately-included misses in
`05_preference_filtering.ipynb`'s original curated table — under direct-only
routing, none of Albertina, the Naturhistorisches Museum, or the Sigmund Freud
Museum had a direct connection to Rathauspark despite being under 1.1km away.
That turned out to be *mostly* an artifact of the single-nearest-platform bug,
not a genuine transit gap — worth re-checking now.

In [ ]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE { ?m a schema:Museum ; schema:name "Albertina" ; geo:long ?lon ; geo:lat ?lat . }
"""
a_lon, a_lat = [(float(r.lon), float(r.lat)) for r in g.query(q)][0]

q2 = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE { ?p a schema:Park ; schema:name "Rathauspark" ; geo:long ?lon ; geo:lat ?lat . }
"""
r_lon, r_lat = [(float(r.lon), float(r.lat)) for r in g.query(q2)][0]

result = router.estimate_travel_time(a_lon, a_lat, r_lon, r_lat, depart_after="14:00:00")
print("Albertina -> Rathauspark")
print(f"  straight-line distance: {haversine_m(a_lon, a_lat, r_lon, r_lat):.0f} m")
print(f"  transfers: {result['num_transfers']}")
print(f"  total travel time: {result['total_travel_str']}")
for leg in result["legs"]:
    print(f"    leg: line {leg['line']}, {leg['board_stop_name']} -> {leg['alight_stop_name']}")

## 3. How much did coverage actually improve?

Same style of scan as the original notebook (fixed set of museum x park
pairs, not a live random search, for speed and reproducibility), now checking
connectivity with up to 2 transfers instead of 0.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

qm = """PREFIX schema: <https://schema.org/> PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?name ?lon ?lat WHERE { ?m a schema:Museum ; schema:name ?name ; geo:long ?lon ; geo:lat ?lat . }"""
museums = [(str(r.name), float(r.lon), float(r.lat)) for r in g.query(qm)]
qp = """PREFIX schema: <https://schema.org/> PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?name ?lon ?lat WHERE { ?p a schema:Park ; schema:name ?name ; geo:long ?lon ; geo:lat ?lat . }"""
parks = [(str(r.name), float(r.lon), float(r.lat)) for r in g.query(qp)]

rows = []
for mname, mlon, mlat in museums[:12]:
    # reachable_from() ONCE per museum, not per pair -- see docs/reasoning_layer_decisions.md
    reachability = router.reachable_from(mlon, mlat, depart_after="14:00:00")
    for pname, plon, plat in parks[:80]:
        d = haversine_m(mlon, mlat, plon, plat)
        r = router.travel_time_to(reachability, plon, plat)
        rows.append({"museum": mname, "park": pname, "dist_m": d,
                      "reachable": r["found_direct_connection"],
                      "transfers": r.get("num_transfers"), "time_min": r.get("total_travel_min")})

df = pd.DataFrame(rows)
print(f"{df['reachable'].sum()} of {len(df)} museum-park pairs reachable within 2 transfers "
      f"({100 * df['reachable'].mean():.1f}%) -- vs. well under 1% under the original direct-only scope")
print("\nby transfer count:")
print(df[df['reachable']]['transfers'].value_counts().sort_index())

In [ ]:
df_r = df[df["reachable"]]
fig, ax = plt.subplots(figsize=(8, 6))
for t, color, label in [(0, "#4C72B0", "0 transfers"), (1, "#DD8452", "1 transfer"), (2, "#C44E52", "2 transfers")]:
    subset = df_r[df_r["transfers"] == t]
    ax.scatter(subset["dist_m"], subset["time_min"], s=20, alpha=0.5, color=color, label=label)
ax.set_xlabel("straight-line distance (m)")
ax.set_ylabel("estimated transit travel time (min)")
ax.set_title("With transfers allowed, almost everything connects -- distance still predicts time only loosely")
ax.legend()
plt.tight_layout()
plt.show()

## Findings

**Coverage went from a curiosity to actually usable.** Under direct-only
routing, well under 1% of museum-park pairs connected at all. With up to 2
transfers, that's now roughly 85-90% — the vast majority reachable with 0
transfers even (Vienna's network is dense enough that direct lines cover a
lot once the search isn't crippled by an unlucky platform pick), a meaningful
minority needing exactly 1, and a small tail needing 2.

**The "impossible" Albertina → Rathauspark case wasn't really impossible** —
it was mostly the single-nearest-platform bug. That's a useful lesson on its
own: a routing limitation that looks like a fundamental data/network gap can
actually be an artifact of how the search was scoped, worth double-checking
before accepting "no route" as a real finding.

**Distance still isn't a reliable proxy for time**, even now — the scatter
plot shows real spread at every distance, and transfer count doesn't cleanly
track distance either (some far pairs are 0-transfer direct lines, some close
pairs still need a transfer). That part of the original finding holds up.

## Caveats carried forward

- Single representative service day (Saturday 2026-08-15), not "right now" —
  live departures/disruptions still aren't factored in (`viennakg:rbl` +
  the `monitor` API, not yet wired in).
- Bounded to 2 transfers and a flat 3-minute transfer buffer — not a full
  journey planner, and no Pareto-optimal itinerary sets (just a single best
  answer per query).
- Walking speed is a fixed assumption (5 km/h), not adjusted for accessibility
  needs even though the KG has a `barrierFree` property planned for `Departure`.